# BART Training — Meeting Summarization

This notebook **trains BART-large-CNN once** and saves the fine-tuned model.

### Project rules
- The existing `dataset/` folder containing meeting audio is **not changed**.
- This notebook does **not** read or modify that audio dataset.
- The final application notebook reads only:
  `outputs/assemblyai/transcripts/*.txt`
- No new folder is created.
- The fine-tuned model weights are saved as one file in the main project folder:
  `bart_model.pt`

### Experiment configuration
- Dataset: **SAMSum**
- Model: **facebook/bart-large-cnn**
- Training examples: **1,000**
- Validation examples: **200**
- Test examples: **100**
- Epochs: **1**
- Maximum input length: **256 tokens**
- Maximum output length: **64 tokens**
- CPU-friendly configuration


In [1]:
# Cell 1 — Install required packages (run once if needed)
%pip install -q transformers datasets accelerate evaluate rouge_score

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Cell 2 — Imports and configuration

import os
import random
import numpy as np
import torch

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

MODEL_NAME = "facebook/bart-large-cnn"
OUTPUT_MODEL_FILE = "bart_model.pt"

TRAIN_SIZE = 1000
VALIDATION_SIZE = 200
TEST_SIZE = 100

MAX_INPUT_LENGTH = 256
MAX_TARGET_LENGTH = 64

print("PyTorch:", torch.__version__)
print("Model:", MODEL_NAME)
print("Device:", "CUDA" if torch.cuda.is_available() else "CPU")


PyTorch: 2.13.0
Model: facebook/bart-large-cnn
Device: CPU


In [3]:
# Cell 3 — Load SAMSum

from datasets import load_dataset

dataset = load_dataset("knkarthick/samsum")

print(dataset)
print("Columns:", dataset["train"].column_names)

print("\nExample:")
print(dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 14731
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 818
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 819
    })
})
Columns: ['id', 'dialogue', 'summary']

Example:
{'id': '13818513', 'dialogue': "Amanda: I baked  cookies. Do you want some?\nJerry: Sure!\nAmanda: I'll bring you tomorrow :-)", 'summary': 'Amanda baked cookies and will bring Jerry some tomorrow.'}


In [4]:
# Cell 4 — Select the documented experiment sizes

train_ds = dataset["train"].shuffle(seed=SEED).select(range(TRAIN_SIZE))
val_ds = dataset["validation"].shuffle(seed=SEED).select(range(VALIDATION_SIZE))
test_ds = dataset["test"].shuffle(seed=SEED).select(range(TEST_SIZE))

print("Training:", len(train_ds))
print("Validation:", len(val_ds))
print("Test:", len(test_ds))


Training: 1000
Validation: 200
Test: 100


In [5]:
# Cell 5 — Load BART-large-CNN

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

print("Tokenizer and model loaded.")


[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

Tokenizer and model loaded.


In [6]:
# Cell 6 — Preprocess dialogue → summary

def preprocess_function(examples):
    inputs = list(examples["dialogue"])
    targets = examples["summary"]

    model_inputs = tokenizer(
        inputs,
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
    )

    labels = tokenizer(
        text_target=targets,
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


tokenized_train = train_ds.map(
    preprocess_function,
    batched=True,
    remove_columns=train_ds.column_names,
)

tokenized_val = val_ds.map(
    preprocess_function,
    batched=True,
    remove_columns=val_ds.column_names,
)

print(tokenized_train)
print(tokenized_val)


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 1000
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 200
})


In [7]:
# Cell 7 — Data collator

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
)

print("Data collator ready.")


Data collator ready.


In [8]:
# Cell 8 — Training configuration

from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir=".",                  # Do not create a new output folder
    num_train_epochs=1,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=5e-5,

    eval_strategy="epoch",
    save_strategy="no",              # Do NOT create checkpoint folders

    logging_strategy="steps",
    logging_steps=100,

    predict_with_generate=True,
    fp16=False,                      # CPU training
    report_to="none",

    load_best_model_at_end=False,
)

print("Training configuration ready.")

Training configuration ready.


In [9]:
# Cell 9 — Create Trainer

from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    processing_class=tokenizer,
)

print("Trainer ready.")

Trainer ready.


In [ ]:
# Cell 10 — Fine-tune BART
# IMPORTANT: this is the expensive cell. Run it once.

import time

start_time = time.time()
train_result = trainer.train()
training_minutes = (time.time() - start_time) / 60

print(f"Training completed in {training_minutes:.2f} minutes.")

/Users/manish/Documents/KU/.venv3.13/lib/python3.13/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


In [ ]:
# Cell 11 — Save the fine-tuned model weights
# No new folder is created. A single .pt file is saved in the project root.

import os
import torch

torch.save(
    trainer.model.state_dict(),
    OUTPUT_MODEL_FILE,
)

print("Fine-tuned BART weights saved to:")
print(os.path.abspath(OUTPUT_MODEL_FILE))


Fine-tuned FLAN-T5 weights saved to:
/Users/manish/Documents/KU/SEM 5/DL/Mini Project/flan_t5_model.pt


In [ ]:
# Cell 12 — Quick test using the saved .pt file

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

base_model_name = "facebook/bart-large-cnn"

test_tokenizer = AutoTokenizer.from_pretrained(base_model_name)
test_model = AutoModelForSeq2SeqLM.from_pretrained(base_model_name)

saved_state = torch.load(OUTPUT_MODEL_FILE, map_location="cpu")
test_model.load_state_dict(saved_state)
test_model.eval()

sample_dialogue = dataset["test"][0]["dialogue"]

inputs = test_tokenizer(
    sample_dialogue,
    return_tensors="pt",
    max_length=MAX_INPUT_LENGTH,
    truncation=True,
)

with torch.no_grad():
    output_ids = test_model.generate(
        **inputs,
        max_length=MAX_TARGET_LENGTH,
        min_length=10,
        num_beams=4,
        do_sample=False,
    )

summary = test_tokenizer.decode(
    output_ids[0],
    skip_special_tokens=True,
)

print("INPUT:")
print(sample_dialogue)
print("\nGENERATED SUMMARY:")
print(summary)


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


INPUT:
Hannah: Hey, do you have Betty's number?
Amanda: Lemme check
Hannah: <file_gif>
Amanda: Sorry, can't find it.
Amanda: Ask Larry
Amanda: He called her last time we were at the park together
Hannah: I don't know him well
Hannah: <file_gif>
Amanda: Don't be shy, he's very nice
Hannah: If you say so..
Hannah: I'd rather you texted him
Amanda: Just text him 🙂
Hannah: Urgh.. Alright
Hannah: Bye
Amanda: Bye bye

GENERATED SUMMARY:
Betty has Betty's number. Amanda asks Larry to call her last time they were at the park together. Larry called her last time they were at the park together.


## Training complete

The important artifact is:

```text
bart_finetuned/
├── config.json
├── generation_config.json
├── model.safetensors (or pytorch_model.bin)
├── tokenizer_config.json
├── tokenizer.json
└── ...
```

No model folder is created.

- `bart_model.pt` = fine-tuned model weights
- `outputs/BART/` = generated meeting summary `.txt` files
